## 🧱 Özel Ortam

---

Bu görevde, derste örnek olarak kullanılan **ızgara tabanlı navigasyon ortamının** birebir aynısını tasarlayıp uygulayacaksınız.

- Ajan ızgaranın bir köşesinde başlar.
- Amaç karşı köşeye ulaşmaktır.
- Izgarada ajanın kaçınması gereken engeller veya "delikler" bulunabilir.

---

### 🎯 Amaçlar

- 📐 **Ortamı Tanımlayın**: ajanın başlangıçtan hedefe doğru hareket ettiği bir ızgara dünyası oluşturun.
- ⚙️ **Ortam Dinamiklerini Uygulayın**: hareket ve ödül ataması kurallarını programlayın (örn. deliklere düşmek için ceza, hedefe ulaşmak için ödül).
- 👁️ **Gözlem ve Eylemleri Ayarlayın**:
    - **Gözlem alanı** → Ajanın algıladığı şeyler (örn. ızgaradaki pozisyonu)  
    - **Eylem alanı** → Ajanın yapabileceği şeyler (örn. yukarı, aşağı, sola, sağa hareket)
- 🖼️ **Render Metodu Ekleyin**: ortamı görselleştirmek için `.render()` fonksiyonu ekleyin — hata ayıklama ve ajanın davranışını anlama için faydalıdır.
- 🧩 **Özel Özellikler Ekleyin**: derste ele alınan engeller, çukurlar veya diğer özellikleri dahil ederek ortamınızı daha dinamik ve gerçekçi hale getirin.

---
Bu görev için ihtiyacımız olan tüm paketleri içe aktararak başlayalım:

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict, Tuple


import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

---

### 🧩 Bölüm 1: Özel Ortam Sınıfı Oluşturma

Bu bölümde, kısmen yazılmış bir sınıfı tamamlayarak özel ortamınızı uygulayacaksınız.  
Yapı, Gymnasium'un ortam oluşturma için standart formatını takip eder.

### 🛠️ Yapacaklarınız

- Bazı temel metotları önceden yazılmış bir şablon sağlanmıştır.  
- `# CODE HERE` yazdığını gördüğünüz her yerde, eksik mantığı doldurmanız gerekir.  
- Her adımda size rehberlik etmesi için satır içi ipuçları verilmiştir.

### 🎯 Hedefiniz

- Sınıfı gerçek bir Gymnasium ortamı gibi davranacak şekilde tamamlayın.  
- Şunları yapmalı:
  - Geçerli bir gözlem ve eylem alanı tanımlamalı  
  - Ajan hareketini ve geçişleri ele almalı  
  - Ödülleri döndürmeli ve `done` bayraklarını uygun şekilde güncellemeli  
  - Çalışan bir `.reset()` ve `.step()` metodu içermeli  
  - İsteğe bağlı olarak görselleştirme için basit bir `.render()` içermeli

🧠 Her metodu anlamak için zaman ayırın — özellikle durum geçişlerinin ve ödüllerin nasıl yönetildiğini. RL ortamları burada canlanır.

📚 Ortam yapısı ve en iyi uygulamalar hakkında ayrıntılı adımlar için resmi [Gymnasium özel ortam kılavuzuna](https://gymnasium.farama.org/introduction/create_custom_env/) başvurun.

In [2]:
class CustomGridEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        # Izgara boyutu: 3x3
        self.size = 3

        # 4 eylem: sağ, yukarı, sol, aşağı
        self.action_space = spaces.Discrete(4)

        # Başlangıç pozisyonu: sol üst köşe
        self.agent_position = np.array([0, 0], dtype=np.int32)

        # Hedef pozisyonu: sağ alt köşe
        self.goal_position = np.array([2, 2], dtype=np.int32)

        # Delik pozisyonu: alt orta
        self.hole_position = np.array([2, 1], dtype=np.int32)

        # Eylemleri hareket yönlerine eşleştir
        self.action_to_direction = {
            0: np.array([0, 1], dtype=np.int32),   # right
            1: np.array([-1, 0], dtype=np.int32),  # up
            2: np.array([0, -1], dtype=np.int32),  # left
            3: np.array([1, 0], dtype=np.int32),   # down
        }

        # Observation space
        self.observation_space = spaces.Dict({
            "agent": spaces.Box(
                low=0,
                high=self.size - 1,
                shape=(2,),
                dtype=np.int32
            ),
            "target": spaces.Box(
                low=0,
                high=self.size - 1,
                shape=(2,),
                dtype=np.int32
            ),
        })

    def reset(
        self,
        seed: Optional[int] = None,
        options: Optional[Dict] = None
    ) -> Tuple[Dict, Dict]:

        # Ortamı sıfırla
        super().reset(seed=seed)

        # Ajanı başlangıç noktasına getir
        self.agent_position = np.array([0, 0], dtype=np.int32)

        # Hedef sabit
        self.goal_position = np.array([2, 2], dtype=np.int32)

        # Observation ve info oluştur
        observation = self._get_obs()
        info = self._get_info()

        return observation, info

    def _get_obs(self):
        # Ajanın ve hedefin mevcut konumlarını döndür
        return {
            "agent": self.agent_position.copy(),
            "target": self.goal_position.copy()
        }

    def _get_info(self):
        # Manhattan mesafesi
        distance = np.sum(
            np.abs(self.agent_position - self.goal_position)
        )

        return {"distance": distance}

    def step(self, action):
        # Seçilen hareket yönünü al
        direction = self.action_to_direction[action]

        # Yeni pozisyonu hesapla
        new_position = self.agent_position + direction

        # Izgaranın dışına çıkmasını engelle
        new_position = np.clip(
            new_position,
            0,
            self.size - 1
        )

        # Ajanın pozisyonunu güncelle
        self.agent_position = new_position.astype(np.int32)

        # Varsayılan değerler
        reward = -1
        done = False

        # Delik kontrolü
        if np.array_equal(
            self.agent_position,
            self.hole_position
        ):
            reward = -10
            done = True

        # Hedef kontrolü
        elif np.array_equal(
            self.agent_position,
            self.goal_position
        ):
            reward = 10
            done = True

        # Observation ve info
        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, done, False, info

    def render(self, mode='human'):

        # Izgarayı oluştur
        grid = np.full(
            (self.size, self.size),
            fill_value=' '
        )

        # Pozisyonları al
        agent_x, agent_y = self.agent_position
        goal_x, goal_y = self.goal_position
        hole_x, hole_y = self.hole_position

        # Izgaraya elemanları yerleştir
        grid[agent_x][agent_y] = 'A'
        grid[goal_x][goal_y] = 'G'
        grid[hole_x][hole_y] = 'H'

        # Izgarayı yazdır
        print("+---" * self.size + "+")
        
        for row in grid:
            print(
                "|" + "|".join(
                    f" {cell} " for cell in row
                ) + "|"
            )
            print("+---" * self.size + "+")

👇 Önceki bölümü doğru tamamladığınızı test etmek için aşağıdaki hücreyi çalıştırın. Doğru yaptıysanız, ajanın bir bölümün sonu olan hedefe veya deliğe ulaşana kadar ortamınızda rastgele hareket ettiğini göreceksiniz.

In [3]:
# Create an instance of the custom environment
env = CustomGridEnv()

# Reset the environment to its initial state and get the initial observation and info
obs, info = env.reset()
print("Initial Observation:", obs)  # Display the initial position of the agent and the target
print("Initial Info:", info)  # Display additional information such as the distance from the target

# Loop through a maximum of 100 steps
for _ in range(100):
    action = env.action_space.sample()  # Randomly sample an action from the action space
    obs, reward, done, info, _ = env.step(action)  # Apply the action and get the results
    env.render()  # Render the current state of the environment to visualize the agent's position

    # Print the current state, reward received, whether the episode is done, and any additional info
    print(f"State: {obs}, Reward: {reward}, Done: {done}, Info: {info}")

    # If the episode is finished (agent reached the goal or fell into a hole), exit the loop
    if done:
        break


Initial Observation: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}
Initial Info: {'distance': 4}
+---+---+---+
|   | A |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 1], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: False
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: False
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: False
+---+---+---+
|   | A |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 1], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: Fal

---
## Bölüm 2: DQN Eğitimi 🤖

Bu bölümde, özel ortamınızı başlatacak ve Stable Baselines3 kütüphanesini kullanarak bir DQN ajanı ile etkileşim için hazırlayacaksınız. Temel görev, ortamınızın kütüphanenin gereksinimleriyle uyumlu olduğundan emin olmaktır, bu da onu uygun şekilde sarmalamayı içerir.

#### 📝 İzlenecek adımlar

1. 🧱 **Özel Ortamı Başlatın**: özel ortam sınıfınızın bir örneğini oluşturun.
2. 🔁 **SB3 Uyumluluğunu Sağlayın**: Stable Baselines3'ten `make_vec_env` fonksiyonunu kullanarak ortamınızı sarın, böylece kütüphanenin vektörleştirilmiş ortam gereksinimleriyle uyumlu hale getirin.
3. ⚙️ **DQN Ajanını Yapılandırın ve Eğitin**: DQN ajanını uygun hiperparametrelerle kurun ve ortamınızda eğitin.
4. 📊 **Eğitim İlerlemesini İzleyin**: Ajanın öğrenme ilerlemesini ve performansını zaman içinde gözlemlemek için günlükleme ve izleme uygulayın.
5. 💾 **Modeli kaydedin**: Eğitim sonrasında modelinizi kaydedin.

In [4]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env

# 1. Özel ortamı oluştur
env = CustomGridEnv()

# 2. SB3 uyumlu vektörleştirilmiş ortama dönüştür
vec_env = make_vec_env(CustomGridEnv, n_envs=1)

# 3. DQN modelini oluştur
model = DQN(
    policy="MultiInputPolicy",
    env=vec_env,
    learning_rate=0.001,
    buffer_size=10000,
    learning_starts=100,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.2,
    exploration_final_eps=0.05,
    verbose=1
)

# 4. Modeli eğit
model.learn(total_timesteps=10000)

# 5. Modeli kaydet
model.save("custom_grid_dqn")

print("Eğitim tamamlandı.")
print("Model kaydedildi: custom_grid_dqn")

Using cpu device
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 22.5     |
|    ep_rew_mean      | -21.5    |
|    exploration_rate | 0.957    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 11361    |
|    time_elapsed     | 0        |
|    total_timesteps  | 90       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 29.9     |
|    ep_rew_mean      | -28.9    |
|    exploration_rate | 0.886    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 2535     |
|    time_elapsed     | 0        |
|    total_timesteps  | 239      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.321    |
|    n_updates        | 34       |
----------------------------------
----------------------------------
| rollout/            |          |
|  

---
## 🎮 Bölüm 3: Eğitilmiş modelinizi kullanın

Eğitilmiş DQN modelinizi yükleyecek ve ortamda karar vermek için kullanacaksınız, başlangıç noktasından hedefe delikleri kaçınarak nasıl gittiğini gözlemleyeceksiniz. Bu sadece eğitimin etkinliğini doğrulamanıza izin vermeyecek, aynı zamanda ajanın karar verme sürecini görsel olarak yorumlamanıza da olanak sağlayacaktır.

### İzlenecek adımlar 📝
1. 💾 **Eğitilmiş Modeli Yükleyin**: Eğitim aşamasında kaydedilen DQN modelini alın.
2. 🔄 **Ortamı Sıfırlayın**: Navigasyon görevini sıfırdan başlatmak için ortamı başlatın.
3. 🧠 **Navigasyon Simülasyonunu Çalıştırın**: Modeli kullanarak ortamın durumlarına dayalı eylemleri tahmin edin ve ajanın attığı her adımı görselleştirin.
4. 👀 **Her Adımı Görselleştirin**: Ajanın pozisyonunu, hedefi ve herhangi bir engeli veya deliği gösteren ızgaranın basit bir görselleştirmesini uygulayın.
5. 📝 **Ajan Davranışını Analiz Edin**: Ajanın hedefe ulaşma yeteneğini gözlemleyin ve not edin ve delikleri ne kadar etkili bir şekilde kaçındığını görün.

Kodun bir kısmı zaten orada ve `# CODE HERE` yorumunu gördüğünüz her yerde kod doldurmanız gerekiyor. Başlamanız için bazı ipuçları bulacaksınız.

In [6]:
# Load the model
model = DQN.load("custom_grid_dqn")

# Reset the environment
obs, info = env.reset()

for _ in range(200):

    # Predict the next action
    action, _states = model.predict(obs, deterministic=True)

    # Convert numpy array to integer
    action = int(action)

    # Execute action
    obs, reward, done, truncated, info = env.step(action)

    # Create 3x3 grid
    grid = np.full((3, 3), fill_value=' ')

    # Get positions
    agent_x, agent_y = obs['agent']
    goal_x, goal_y = obs['target']
    hole_x, hole_y = 2, 1

    # Update grid
    grid[agent_x][agent_y] = 'A'
    grid[goal_x][goal_y] = 'G'
    grid[hole_x][hole_y] = 'H'

    # Print state
    print(
        f"State: {obs}, "
        f"Reward: {reward}, "
        f"Done: {done}, "
        f"Info: {info}"
    )

    # Print grid
    print("+---" * 3 + "+")

    for row in grid:
        print("|" + "|".join(f" {cell} " for cell in row))
        print("+---" * 3 + "+")

    # Reset if episode finished
    if done or truncated:
        print("Episode finished. Resetting...")
        obs, info = env.reset()

State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': 4}
+---+---+---+
| A |   |   
+---+---+---+
|   |   |   
+---+---+---+
|   | H | G 
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': 4}
+---+---+---+
| A |   |   
+---+---+---+
|   |   |   
+---+---+---+
|   | H | G 
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': 4}
+---+---+---+
| A |   |   
+---+---+---+
|   |   |   
+---+---+---+
|   | H | G 
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': 4}
+---+---+---+
| A |   |   
+---+---+---+
|   |   |   
+---+---+---+
|   | H | G 
+---+---+---+
State: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}, Reward: -1, 

🧠 Artık ajanınızın ortamda hareket ettiğini görmelisiniz.

Eğer ajan **belirli eylemleri tekrarlayarak takılıp kalırsa** veya hedefe ulaşamazsa, muhtemelen **yeterince uzun eğitilmediği** içindir.

Eğitim adım sayısını artırmayı deneyin ve modeli yeniden eğitin — daha uzun eğitim genellikle ajanın daha iyi bir politika öğrenmesine yardımcı olur.